# ⚖️ Vietnam Legal Agent - SOTA Vector Indexer (`darklethelong/vnlegal-lal`)
Index toàn bộ **318 Bộ luật & Luật Quốc gia** + **67.000+ điều Bộ Pháp điển** vào Qdrant Vector DB với model SOTA `darklethelong/vnlegal-lal` (1024-dim, FP16 CUDA).

In [ ]:
# 1. Install dependencies
!pip install -q sentence-transformers qdrant-client pyarrow tqdm

import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    print("Using CPU")
    DEVICE = "cpu"
    torch.set_num_threads(8)

In [ ]:
# 2. Load Datasets: 318 National Laws + PhapDien (67,000+ Articles)
import os, sys, time, re, urllib.request, tarfile
import pyarrow.parquet as pq

articles = []
CORPUS_DIR = '/tmp/data_corpus'
os.makedirs(CORPUS_DIR, exist_ok=True)

# 2.1 Load UTS_VLC (318 Laws)
uts_path = os.path.join(CORPUS_DIR, 'uts_vlc_2026_01.parquet')
if not os.path.exists(uts_path):
    for cand in ['uts_vlc_2026_01.parquet', '/kaggle/working/uts_vlc_2026_01.parquet', 'kaggle_indexer/uts_vlc_2026_01.parquet']:
        if os.path.exists(cand):
            import shutil; shutil.copy(cand, uts_path)
            break

if os.path.exists(uts_path):
    print('Loading UTS_VLC...')
    table = pq.read_table(uts_path).to_pydict()
    art_split_pattern = re.compile(r'(?=(?:^|\n)(?:###?\s*)?Điều\s+\d+[\w\.]*\.?\s*)', re.MULTILINE)
    num_laws = len(table.get('id', []))
    for i in range(num_laws):
        law_id = str(table['id'][i])
        law_title = str(table.get('title', [''])[i] or '')
        content = str(table.get('content', [''])[i] or '')
        domain = str(table.get('domain', ['Luật Quốc gia'])[i] or 'Luật Quốc gia')
        status = str(table.get('status', ['Còn hiệu lực'])[i] or 'Còn hiệu lực')
        code = str(table.get('code', [''])[i] or '')
        split_arts = art_split_pattern.split(content)
        for idx, art in enumerate(split_arts[1:], 1):
            art_clean = art.strip()
            if not art_clean:
                continue
            articles.append({
                'record_id': f'{law_id}-art-{idx}',
                'topic': 'Luật Quốc gia',
                'subject': domain,
                'document_title': law_title,
                'document_code': code,
                'article_title': art_clean.splitlines()[0][:140],
                'effective_status': status,
                'content_text': art_clean,
            })
    print(f'Loaded {len(articles):,} articles from 318 National Laws.')

# 2.2 Download & Load PhapDien (7 Parquet Shards)
print('Downloading PhapDien shards...')
HF_BASE = 'https://huggingface.co/datasets/tmquan/phapdien-moj-gov-vn/resolve/main'
for i in range(7):
    fname = f'articles-{i:05d}-of-00007.parquet'
    lpath = os.path.join(CORPUS_DIR, fname)
    if not os.path.exists(lpath):
        urllib.request.urlretrieve(f'{HF_BASE}/{fname}', lpath)
    t = pq.read_table(lpath).to_pydict()
    for j in range(len(t['record_id'])):
        articles.append({
            'record_id': str(t['record_id'][j]),
            'topic': str(t['topic_title_vi'][j] or 'Pháp điển'),
            'subject': str(t['subject_title_vi'][j] or ''),
            'document_title': str(t['source_note_text'][j] or 'Bộ Pháp điển Việt Nam'),
            'document_code': '',
            'article_title': str(t['article_title'][j] or ''),
            'effective_status': 'Còn hiệu lực',
            'content_text': str(t['content_text'][j] or ''),
        })
print(f'✅ Total Legal Articles to Index: {len(articles):,}')

In [ ]:
# 3. Load SOTA Embedding Model: darklethelong/vnlegal-lal
from sentence_transformers import SentenceTransformer
MODEL_NAME = 'darklethelong/vnlegal-lal'
print(f'=== Loading Model: {MODEL_NAME} on {DEVICE} ===', flush=True)

model = SentenceTransformer(MODEL_NAME, device=DEVICE)
if DEVICE == 'cuda':
    model.half()
    print('Enabled FP16 precision on CUDA.', flush=True)

VECTOR_DIM = 1024

In [ ]:
# 4. Initialize Local Qdrant Database
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, HnswConfigDiff, PointStruct

OUTPUT_DIR = '/tmp/qdrant_db'
os.makedirs(OUTPUT_DIR, exist_ok=True)

client = QdrantClient(path=OUTPUT_DIR)
COLLECTION_NAME = 'vietnam_legal_collection_v1'

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
    hnsw_config=HnswConfigDiff(m=16, ef_construct=128, full_scan_threshold=10000),
)
print(f'✅ Qdrant Collection {COLLECTION_NAME} initialized.', flush=True)

In [ ]:
# 5. Batch Encoding with Live Progress (~3-4 minutes on GPU T4)
BATCH_SIZE = 64 if DEVICE == 'cuda' else 32
texts_to_embed = [
    f"Văn bản: {a['document_title']}\nChủ đề: {a['topic']} - {a['subject']}\nĐiều: {a['article_title']}\nNội dung: {a['content_text'][:3500]}"
    for a in articles
]

total_records = len(articles)
started_at = time.time()
print(f'=== Starting Batch Encoding of {total_records:,} items (Batch Size: {BATCH_SIZE}) ===', flush=True)

for batch_num, idx in enumerate(range(0, total_records, BATCH_SIZE), 1):
    batch_texts = texts_to_embed[idx : idx + BATCH_SIZE]
    batch_articles = articles[idx : idx + BATCH_SIZE]
    
    with torch.no_grad():
        embeddings = model.encode(
            batch_texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            normalize_embeddings=True,
        )
    
    points = [
        PointStruct(
            id=idx + i + 1,
            vector=embeddings[i].tolist(),
            payload={
                'record_id': item['record_id'],
                'topic': item['topic'],
                'subject': item['subject'],
                'document_title': item['document_title'],
                'document_code': item['document_code'],
                'article_title': item['article_title'],
                'effective_status': item['effective_status'],
                'source': item['document_title'],
                'text': item['content_text'][:2000],
            }
        )
        for i, item in enumerate(batch_articles)
    ]
    
    client.upsert(collection_name=COLLECTION_NAME, points=points)
    
    if batch_num % 5 == 0 or idx + BATCH_SIZE >= total_records:
        current_count = min(idx + len(batch_articles), total_records)
        elapsed = time.time() - started_at
        speed = current_count / max(elapsed, 0.1)
        remaining = (total_records - current_count) / max(speed, 0.1)
        pct = (current_count / total_records) * 100
        print(f'⚡ [{current_count:,}/{total_records:,}] ({pct:.1f}%) | Speed: {speed:.1f} vec/s | Elapsed: {elapsed:.0f}s | ETA: {remaining:.0f}s', flush=True)

duration = time.time() - started_at
print(f'🎉 Completed indexing {total_records:,} vectors in {duration:.1f}s (Avg speed: {total_records/duration:.1f} vec/s)', flush=True)

In [ ]:
# 6. Compress and Download
artifact_name = 'qdrant_vnlegal_lal.tar.gz'
print(f'Compressing {OUTPUT_DIR} into {artifact_name}...', flush=True)
with tarfile.open(artifact_name, 'w:gz') as tar:
    tar.add(OUTPUT_DIR, arcname='qdrant_db')
size_mb = os.path.getsize(artifact_name) / (1024 * 1024)
print(f'✅ Artifact created: {artifact_name} ({size_mb:.1f} MB)', flush=True)

# Auto download if in Google Colab
try:
    from google.colab import files
    files.download(artifact_name)
except Exception:
    pass